# Chapter 03 — CSV, JSON, and Excel Data Handling

Files are common boundaries between systems. Learn text encoding, CSV delimiters and quoting, JSON objects and arrays, Excel workbooks, and robust import/export patterns.

## Prerequisites
Python file handling and basic DataFrame concepts.

## Learning goals
- Read and write CSV with the standard library and Pandas.
- Parse JSON objects and arrays and understand serialization options.
- Import/export Excel workbooks with Pandas and OpenPyXL.
- Handle encodings, missing values, and data types.

**Study sequence:** use the theory notebook first, then tutorial, then lab. The examples use the bundled synthetic data unless otherwise noted.

In [1]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())

Data directory: /mnt/data/python_data_analysis_book/data


## Guided learning path

Work from top to bottom. Each step states the purpose, explains the API choice, and provides a complete code cell. Modify parameters and predict the output before running it.

### Step 1 — Guided 1 — Validate CSV columns

A successful file read does not prove the schema is correct. Compare the imported column names with the required contract.

In [2]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())
import pandas as pd
sales=pd.read_csv(DATA/"sales.csv",dtype={"order_id":"string"},parse_dates=["order_date"])
required={"order_id","order_date","region","revenue"}
missing=required-set(sales.columns)
if missing: raise ValueError(f"Missing columns: {sorted(missing)}")
print("Schema passed",sales.shape)

Data directory: /mnt/data/python_data_analysis_book/data
Schema passed (240, 11)


### Step 2 — Guided 2 — Load JSON

`json.loads` expects a string; `json.load` expects a file-like object. `DataFrame.from_records` makes the row-oriented conversion explicit.

In [3]:
import json, pandas as pd
records=json.loads((DATA/"weather.json").read_text(encoding="utf-8"))
weather=pd.DataFrame.from_records(records); display(weather)

         date     city  temperature_c  humidity_pct condition
0  2025-01-01  Kolkata           22.5            68     Clear
1  2025-01-02  Kolkata           23.1            65    Cloudy
2  2025-01-03    Delhi           14.2            42       Fog
3  2025-01-04   Mumbai           28.4            76     Humid


### Step 3 — Guided 3 — Select and export data

`usecols` restricts import columns; `index=False` prevents a DataFrame index from becoming an unwanted output column.

In [4]:
report=sales.groupby("region",as_index=False).agg(orders=("order_id","count"),revenue=("revenue","sum"))
report.to_csv("region_report.csv",index=False,encoding="utf-8")
report.to_json("region_report.json",orient="records",indent=2)
display(report)

  region  orders      revenue
0   East      63   8707069.76
1  North      53   8161814.90
2  South      58  10179983.56
3   West      66  12370641.44


### Step 4 — Guided 4 — Create Excel sheets

`ExcelWriter` is a context manager; it closes the workbook after all sheets are written.

In [5]:
with pd.ExcelWriter("region_report.xlsx",engine="openpyxl") as writer:
 report.to_excel(writer,sheet_name="Regions",index=False)
 weather.to_excel(writer,sheet_name="Weather",index=False)
print(pd.ExcelFile("region_report.xlsx").sheet_names)

['Regions', 'Weather']


## Tutorial checkpoint

1. What was the input data type?
2. What did the function or method return?
3. Which argument most changed the result?
4. What edge case should be tested before using this in production?